# Simulation examples

This notebook shows the current simulator API in two settings:

- one episode, step by step
- many episodes, with density mean and standard deviation


## Imports


In [ ]:
import sys
from pathlib import Path

import igraph as ig
import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd()
if not (project_root / "network_creation").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    scienceplots = None
else:
    plt.style.use("science")

from network_creation.core.agent import Agent
from network_creation.core.environment import Environment
from network_creation.core.graph import create_empty_graph
from network_creation.core.model import ActorScheduler
from network_creation.core.observability import ObservationSampling
from network_creation.core.simulator import Simulator
from network_creation.observers.grimoire import OBSERVER_GRIMOIRE
from network_creation.plotting.episode_series import plot_episode_series_observer
from network_creation.plotting.graph_history import (
    build_circle_layout,
    plot_graph_history,
)
from network_creation.time_normalization import normalized_source_times
from network_creation.policies.random import RandomPolicy
from network_creation.policies.best_response import BestResponsePolicy
from network_creation.plotting.plot_network import draw_network

ActionPrintObserver = OBSERVER_GRIMOIRE["action_print"]
DensityObserver = OBSERVER_GRIMOIRE["density"]
StateHistoryObserver = OBSERVER_GRIMOIRE["state_history"]
CenteredMeanWelfareObserver = OBSERVER_GRIMOIRE["centered_mean_welfare"]
CentralPointOfDominanceObserver = OBSERVER_GRIMOIRE["central_point_of_dominance"]

## Single episode


In [ ]:
initial_graph_adj = np.array(
    [
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0],
    ],
    dtype=np.int8,
)
init_graph = ig.Graph.Adjacency(initial_graph_adj.tolist(), mode=ig.ADJ_DIRECTED)
fixed_layout = build_circle_layout(n_nodes=initial_graph_adj.shape[0])

env = Environment(graph=init_graph, seed=11)
fixed_layout = build_circle_layout(n_nodes=initial_graph_adj.shape[0])

agents = tuple(
    Agent(
        id=node_id,
        visibility_distance=2,
        policy=BestResponsePolicy(
            n_nodes=initial_graph_adj.shape[0],
            seed=11 + node_id,
            benefit_to_cost_ratio=0.5,
            distance_discount=0.9,
            outside_optimism=0.0,
            exploration=0.54,
        ),
        seed=42 + node_id,
    )
    for node_id in range(initial_graph_adj.shape[0])
)

max_steps = 12
simulator_seed = 301
actor_scheduler = ActorScheduler(
    n_agents=initial_graph_adj.shape[0], n_actors=1, seed=simulator_seed
)

action_printer = ActionPrintObserver(
    observation_sampling=ObservationSampling.periodic(1), print_empty_steps=False
)
state_history_observer = StateHistoryObserver(
    observation_sampling=ObservationSampling.periodic(1), keep_runs=False
)

In [ ]:
simulator = Simulator(
    env=env,
    agents=agents,
    actor_scheduler=actor_scheduler,
    max_steps=max_steps,
    rng=np.random.default_rng(simulator_seed),
    observers=(action_printer, state_history_observer),
)

simulator.run(n_episodes=1, reset_observers=False)

### Directed graph after each sampled step


In [ ]:
plot_graph_history(
    steps=state_history_observer.steps,
    graphs=state_history_observer.graphs,
    layout=fixed_layout,
    figure_title="Directed graph over one episode",
    max_cols=4,
)
plt.show()

## Multiple episodes

The simulator now derives a fresh seed for each agent at each episode.
That keeps repeated runs reproducible while avoiding identical random episodes.


In [ ]:
n_agents = 10
multi_n_actors = 1
benefit_to_cost_ratio = 10 / (2 * n_agents)
distance_discount = 0.9

multi_env = Environment(graph=create_empty_graph(n_agents=n_agents), seed=101)
multi_agents = tuple(
    Agent(
        id=node_id,
        visibility_distance=6,
        policy=BestResponsePolicy(
            n_nodes=n_agents,
            seed=11 + node_id,
            benefit_to_cost_ratio=benefit_to_cost_ratio,
            distance_discount=distance_discount,
            outside_optimism=0.43,
            exploration=0.020,
        ),
        seed=500 + node_id,
    )
    for node_id in range(multi_env.state.graph.vcount())
)
multi_wellfare_observer = CenteredMeanWelfareObserver(
    n_agents=n_agents,
    observation_sampling=ObservationSampling.periodic(1),
    aggregations=("mean", "std", "runs"),
    benefit_to_cost_ratio=benefit_to_cost_ratio,
    distance_discount=distance_discount,
)
state_history_observer = StateHistoryObserver(
    observation_sampling=ObservationSampling.periodic(1), keep_runs=True
)
central_point_of_dominance_observer = CentralPointOfDominanceObserver(
    n_agents=n_agents,
    observation_sampling=ObservationSampling.periodic(1),
    aggregations=("mean", "std", "runs"),
)
action_printer = ActionPrintObserver(
    observation_sampling=ObservationSampling.periodic(1), print_empty_steps=False
)

multi_simulator = Simulator(
    env=multi_env,
    agents=multi_agents,
    actor_scheduler=ActorScheduler(
        n_agents=len(multi_agents), n_actors=multi_n_actors, seed=41
    ),
    max_steps=6000,
    rng=np.random.default_rng(41),
    observers=(
        multi_wellfare_observer,
        state_history_observer,
        central_point_of_dominance_observer,
        # action_printer
    ),
    stagnation_patience=None,
)

multi_simulator.run(n_episodes=16, reset_observers=True)
multi_welfare_result = multi_wellfare_observer.get()
state_history_result = state_history_observer.get()

multi_welfare_result.n_episodes, tuple(multi_welfare_result.outputs)

In [ ]:
figure, axis = plt.subplots(figsize=(12, 5))
normalized_steps = normalized_source_times(
    multi_wellfare_observer.steps,
    n_agents=n_agents,
    n_actors=multi_n_actors,
)
ax = plot_episode_series_observer(
    observer=multi_wellfare_observer,
    ax=axis,
    show_runs=True,
    x_values=normalized_steps,
    x_label="activation opportunities per agent",
)
plt.show()

In [ ]:
nb_subplots = 4
graphs = [
    g_t[-1].to_networkx(vertex_attr_hashable="name")
    for g_t in state_history_observer._runs_graphs[:nb_subplots]
]

ncols = 2
nrows = int(np.ceil(nb_subplots / ncols))
fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols, figsize=(8 * ncols, 8 * nrows), dpi=120
)

axes = np.ravel(axes)

for graph, ax in zip(graphs, axes):
    res = draw_network(graph=graph, ax=ax)
    ax.patch.set_edgecolor("black")
    ax.patch.set_linewidth(1)

for ax in axes[len(graphs) :]:
    ax.set_visible(False)  # or: fig.delaxes(ax)

plt.show()

In [ ]:
num_runs = 10
fixed_layout = build_circle_layout(n_nodes=n_agents)
plot_graph_history(
    steps=np.arange(num_runs),
    graphs=[graphs_run[-1] for graphs_run in state_history_observer._runs_graphs][
        :num_runs
    ],
    layout=fixed_layout,
    figure_title="",
    max_cols=4,
)
plt.show()

# interactive plot


In [ ]:
from netgraph import Graph
import networkx as nx
import matplotlib as mpl
from netgraph import InteractiveGraph

In [ ]:
g = state_history_result.graphs[-1].to_networkx(vertex_attr_hashable="name")

In [ ]:
%matplotlib ipympl

_, ax = plt.subplots(nrows=1, ncols=1, figsize=(8,8), dpi=120)

node_labels = {n: str(n) for n in g.nodes()}
plot_instance = InteractiveGraph(
    g, 
    node_labels=node_labels, 
    edge_label_fontdict=dict(size=8, fontweight='normal', color="b", bbox=dict(boxstyle='round', ec=(1.0, 1.0, 1.0), fc=(1.0, 1.0, 1.0), alpha=0.9)),
    node_layout='spring', 
    node_layout_kwargs={'pad_by': 0.01, "k":0.3},
    edge_layout='curved',
    node_shape='o',
    node_label_fontdict = dict(size=8, fontweight='normal'),
    node_size=3, 
    edge_width=0.5, 
    edge_color='#d111b7',
    arrows=True, 
    ax=ax)

plt.show()